# 04 Label Sentiment

This notebook creates and manually labels the fixed sentiment-evaluation subset.


## Fixed sample

The submitted specification requires a fixed manually labelled subset but does not prescribe its size. For this MVP, the notebook selects **300 headlines**: 30 randomly sampled headlines from each of the ten companies. The fixed random seed `2025` makes the selection reproducible and company balancing prevents firms with more GDELT coverage from dominating the labelled subset. A sampled headline that turns out to concern a different company is replaced by another headline for the same company rather than labelled neutral.

Labels are assigned to the target company using only information explicitly stated in the headline. This follows the entity-aware, investor-perspective approach described by [Sinha et al. (2022)](https://doi.org/10.1002/asi.24634).

In [1]:
from pathlib import Path
import hashlib

import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [2]:
# Support running the notebook from either the project root or the pipeline directory.
working_directory = Path.cwd()
project_root = (
    working_directory.parent
    if working_directory.name == "pipeline"
    else working_directory
)

aligned_path = project_root / "data" / "processed" / "headlines_aligned_prices.csv"
gold_labels_path = project_root / "data" / "processed" / "gold_labels.csv"

print("Project root:", project_root)
print("Aligned data exists:", aligned_path.exists())


Project root: C:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks
Aligned data exists: True


In [3]:
aligned_df = pd.read_csv(aligned_path)

required_columns = {
    "headline_id",
    "published_at_utc",
    "published_at_london",
    "ticker",
    "company_name",
    "headline_text",
}

assert required_columns.issubset(aligned_df.columns)
assert aligned_df["headline_id"].is_unique
assert not aligned_df[list(required_columns)].isna().any().any()
assert aligned_df["ticker"].nunique() == 10

print(f"Available aligned headlines: {len(aligned_df):,}")


Available aligned headlines: 9,001


In [4]:
SAMPLE_PER_COMPANY = 30
RANDOM_SEED = 2025
# The row-numbered labels below only apply to this exact sample.
SAMPLE_SHA256 = "3d70b333c232941367fcbdfe82bc2bf4c0140190fa780be61bec4547aaa523bb"

assert aligned_df.groupby("ticker").size().min() >= SAMPLE_PER_COMPANY

sample_df = (
    aligned_df.groupby("ticker", group_keys=False)
    .sample(n=SAMPLE_PER_COMPANY, random_state=RANDOM_SEED)
    .sort_values(["ticker", "published_at_utc", "headline_text"])
    .reset_index(drop=True)
)

assert len(sample_df) == 300
assert sample_df.groupby("ticker").size().eq(SAMPLE_PER_COMPANY).all()
assert sample_df["headline_id"].is_unique
assert hashlib.sha256("\n".join(sample_df["headline_id"]).encode("utf-8")).hexdigest() == SAMPLE_SHA256, (
    "The sample has changed, so the row numbers used for labelling no longer match."
)

display(sample_df.groupby("ticker").size().rename("sampled_headlines"))

ticker
AZN.L     30
GSK.L     30
HLMA.L    30
HSBA.L    30
ITRK.L    30
NG.L      30
REL.L     30
RIO.L     30
SN.L      30
VOD.L     30
Name: sampled_headlines, dtype: int64

In [5]:
# Row numbers of sampled headlines that are about a different company. Each one is
# replaced by an unused headline for the same ticker (added as row 300 onwards)
# instead of being labelled. Only append to this list, because its order fixes the
# row numbers of the replacements.
WRONG_COMPANY_ROWS = []

# The unsampled headlines in a fixed random order, used as replacements.
reserve_df = aligned_df.loc[
    ~aligned_df["headline_id"].isin(sample_df["headline_id"])
].sample(frac=1, random_state=RANDOM_SEED)

initial_sample_size = len(sample_df)
sample_df["replaced"] = False

for row_number in WRONG_COMPANY_ROWS:
    assert not sample_df.loc[row_number, "replaced"], f"Row {row_number} is listed twice."
    ticker = sample_df.loc[row_number, "ticker"]
    unused_headlines = reserve_df.loc[
        (reserve_df["ticker"] == ticker)
        & ~reserve_df["headline_id"].isin(sample_df["headline_id"])
    ]
    sample_df.loc[row_number, "replaced"] = True
    sample_df = pd.concat(
        [sample_df, unused_headlines.head(1).assign(replaced=False)],
        ignore_index=True,
    )

print(f"Wrong-company headlines replaced: {len(WRONG_COMPANY_ROWS)}")

Wrong-company headlines replaced: 0


## Labelling guide

Judge the likely financial effect on the **named target company**, using only the headline. Do not use the later share price, article body, or outside knowledge.

- **positive**: the headline clearly indicates a favourable development, such as improved results, an approval, a contract win, an upgrade, or another likely benefit.
- **negative**: the headline clearly indicates an unfavourable development, such as weaker results, a warning, a rejection, litigation, disruption, a downgrade, or another likely harm.
- **neutral**: the headline is factual, mixed, unclear, or does not indicate a clear positive or negative effect on the company.



In [6]:
display_columns = [
    "headline_id",
    "ticker",
    "company_name",
    "published_at_london",
    "headline_text",
    "replaced",
]
GOLD_COLUMNS = ["headline_id", "label_gold", "annotator_pass", "qa_flag"]

labelling_df = sample_df[display_columns].copy()

if gold_labels_path.exists():
    existing_labels = pd.read_csv(gold_labels_path, keep_default_na=False)
    # Files saved before qa_flag was added only have the first three columns.
    if "qa_flag" not in existing_labels.columns:
        existing_labels["qa_flag"] = 0
    assert existing_labels.columns.tolist() == GOLD_COLUMNS
    assert existing_labels["headline_id"].is_unique
    assert set(existing_labels["headline_id"]).issubset(
        set(labelling_df["headline_id"])
    )
    labelling_df = labelling_df.merge(
        existing_labels, on="headline_id", how="left"
    )
    labelling_df["label_gold"] = labelling_df["label_gold"].fillna("")
    labelling_df["annotator_pass"] = (
        labelling_df["annotator_pass"].fillna(1).astype(int)
    )
    labelling_df["qa_flag"] = labelling_df["qa_flag"].fillna(0).astype(int)
else:
    labelling_df["label_gold"] = ""
    labelling_df["annotator_pass"] = 1
    labelling_df["qa_flag"] = 0

print("Existing label progress loaded.")

Existing label progress loaded.


## Initial labels

The cells below assign a headline-only label to every sampled headline. They are separated by ticker so the decisions are easy to inspect. Rerunning these cells fills only blank labels and does not overwrite labels already saved in `gold_labels.csv`.


In [7]:
# The first-pass decisions, kept so the second pass can be compared with them.
first_pass_labels = pd.Series("", index=labelling_df.index)


def assign_labels(indices, label):
    """Assign a label to blank rows without overwriting saved work."""

    first_pass_labels.loc[indices] = label
    rows_are_selected = labelling_df.index.isin(indices)
    labels_are_blank = labelling_df["label_gold"] == ""
    rows_to_label = rows_are_selected & labels_are_blank

    labelling_df.loc[rows_to_label, "label_gold"] = label
    labelling_df.loc[rows_to_label, "annotator_pass"] = 1

In [8]:
# AstraZeneca (AZN.L): rows 0-29
assign_labels(
    [0, 1, 2, 6, 9, 10, 12, 16, 17, 18, 19, 23, 25, 27, 28, 29],
    "positive",
)
assign_labels([3, 4, 7, 8, 11, 14, 15, 21, 24], "negative")
assign_labels([5, 13, 20, 22, 26], "neutral")


In [9]:
# GSK (GSK.L): rows 30-59
assign_labels(
    [31, 33, 35, 36, 37, 38, 41, 43, 44, 45, 46, 47, 50, 52, 56, 57],
    "positive",
)
assign_labels([34, 39, 40, 42, 49, 55, 58], "negative")
assign_labels([30, 32, 48, 51, 53, 54, 59], "neutral")


In [10]:
# Halma (HLMA.L): rows 60-89
assign_labels(
    [64, 65, 66, 67, 68, 69, 70, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 83, 84, 88],
    "positive",
)
assign_labels([62, 78, 85], "negative")
assign_labels([60, 61, 63, 71, 86, 87, 89], "neutral")


In [11]:
# HSBC (HSBA.L): rows 90-119
assign_labels([90, 92, 93, 96, 97, 100, 103, 111, 115], "positive")
assign_labels([94, 95, 99, 102, 109], "negative")
assign_labels(
    [91, 98, 101, 104, 105, 106, 107, 108, 110, 112, 113, 114, 116, 117, 118, 119],
    "neutral",
)


In [12]:
# Intertek (ITRK.L): rows 120-149
assign_labels(
    [120, 121, 122, 123, 124, 125, 126, 130, 132, 133, 135, 136, 137, 141, 142, 143, 145, 146, 148],
    "positive",
)
assign_labels([128, 129, 134, 140, 144], "negative")
assign_labels([127, 131, 138, 139, 147, 149], "neutral")


In [13]:
# National Grid (NG.L): rows 150-179
assign_labels([157, 162, 163, 165, 170, 171, 177, 179], "positive")
assign_labels([150, 152, 153, 158, 159, 164, 166, 167, 168], "negative")
assign_labels(
    [151, 154, 155, 156, 160, 161, 169, 172, 173, 174, 175, 176, 178],
    "neutral",
)


In [14]:
# RELX (REL.L): rows 180-209
assign_labels(
    [180, 181, 182, 183, 184, 187, 189, 190, 191, 192, 194, 195, 196, 201, 202, 203, 204, 206, 207, 208],
    "positive",
)
assign_labels([185, 186, 193, 197, 209], "negative")
assign_labels([188, 198, 199, 200, 205], "neutral")


In [15]:
# Rio Tinto (RIO.L): rows 210-239
assign_labels(
    [215, 216, 219, 220, 221, 225, 227, 230, 232, 233, 237, 238, 239],
    "positive",
)
assign_labels([210, 217, 218, 223, 224, 226, 228, 236], "negative")
assign_labels([211, 212, 213, 214, 222, 229, 231, 234, 235], "neutral")


In [16]:
# Smith & Nephew (SN.L): rows 240-269
assign_labels(
    [240, 241, 242, 243, 244, 246, 247, 248, 253, 259, 260, 261, 262, 264, 265, 266, 267, 268],
    "positive",
)
assign_labels([249, 252, 254, 269], "negative")
assign_labels([245, 250, 251, 255, 256, 257, 258, 263], "neutral")


In [17]:
# Vodafone (VOD.L): rows 270-299
assign_labels(
    [271, 273, 275, 277, 283, 285, 286, 287, 288, 289, 294, 296, 297],
    "positive",
)
assign_labels([272, 274, 276, 280, 281, 282, 291, 292, 295, 298], "negative")
assign_labels([270, 278, 279, 284, 290, 293, 299], "neutral")


In [18]:
# Replacement headlines: rows 300 onwards, created by WRONG_COMPANY_ROWS.
# assign_labels([300], "neutral")

## Second pass

Change `TICKER_TO_INSPECT` and run the display cell to reread one company at a time. Then:

- record every label you change in `SECOND_PASS_CORRECTIONS`, using the displayed row number;
- add headlines about a different company to `WRONG_COMPANY_ROWS` near the top of the notebook, rerun from there, and label the replacement rows instead;
- set `SECOND_PASS_COMPLETE = True` once all ten companies have been checked, and run the save cell.

Rows whose final label differs from the first pass get `qa_flag = 1`.

In [19]:
TICKER_TO_INSPECT = "AZN.L"

display(
    labelling_df.loc[
        labelling_df["ticker"] == TICKER_TO_INSPECT,
        [
            "ticker",
            "company_name",
            "headline_text",
            "label_gold",
            "annotator_pass",
            "qa_flag",
            "replaced",
        ],
    ]
)

,ticker,company_name,headline_text,label_gold,annotator_pass,qa_flag,replaced
0,AZN.L,AstraZeneca plc,Trinasolar Powers AstraZeneca's Sustainability Vision with Cutting-Edge Solar Carpark Rooftop Install,positive,1,0,False
1,AZN.L,AstraZeneca plc,Trinasolar Powers AstraZeneca's Sustainability Vision with Cutting-Edge Solar Carpark Rooftop Install | Taiwan News,positive,1,0,False
2,AZN.L,AstraZeneca plc,ASTRAZENECA RECEIVES TWO POSITIVE NICE RECOMMENDATIONS FOR LUNG CANCER PATIENTS ACROSS ENGLAND AND WALES,positive,1,0,False
3,AZN.L,AstraZeneca plc,AstraZeneca accused of rowing back on pledges in vaccine hub row,negative,1,0,False
4,AZN.L,AstraZeneca plc,Covid chief attacks Labour after AstraZeneca axes its plan for £450million vaccine site,negative,1,0,False
5,AZN.L,AstraZeneca plc,"Bank of England, AstraZeneca, Watches of Switzerland: Thursday ahead",neutral,1,0,False
6,AZN.L,AstraZeneca plc,AstraZeneca: A strong pipeline is a big tailwind - London Business News,positive,1,0,False
7,AZN.L,AstraZeneca plc,Investors in AstraZeneca PLC Should Contact Levi & Korsinsky Before ...,negative,1,0,False
8,AZN.L,AstraZeneca plc,Here's Why AstraZeneca PLC (AZN) Traded Lower in Q4,negative,1,0,False
9,AZN.L,AstraZeneca plc,"AstraZeneca Pharma India gets CDSCO approval to import, sell cancer treatment medicine",positive,1,0,False


In [20]:
# Labels changed in the second pass, keyed by displayed row number, e.g. {12: "neutral"}.
SECOND_PASS_CORRECTIONS = {}
SECOND_PASS_COMPLETE = False

assert set(SECOND_PASS_CORRECTIONS.values()).issubset({"positive", "neutral", "negative"})
assert not labelling_df.loc[list(SECOND_PASS_CORRECTIONS), "replaced"].any()

for row_number, label in SECOND_PASS_CORRECTIONS.items():
    labelling_df.loc[row_number, "label_gold"] = label

labelling_df["qa_flag"] = (
    (first_pass_labels != "") & (labelling_df["label_gold"] != first_pass_labels)
).astype(int)

if SECOND_PASS_COMPLETE:
    labelling_df.loc[labelling_df["label_gold"] != "", "annotator_pass"] = 2

In [21]:
valid_labels = {"positive", "neutral", "negative"}
kept_df = labelling_df.loc[~labelling_df["replaced"]]

entered_labels = set(kept_df["label_gold"])
invalid_labels = entered_labels - valid_labels - {""}

assert not invalid_labels, f"Invalid labels: {sorted(invalid_labels)}"
assert kept_df["annotator_pass"].isin([1, 2]).all()
assert kept_df.groupby("ticker").size().eq(SAMPLE_PER_COMPANY).all()

unlabelled_count = int((kept_df["label_gold"] == "").sum())

print(f"Labelled: {len(kept_df) - unlabelled_count:,} / {len(kept_df):,}")
print(f"Wrong-company headlines replaced: {int(labelling_df['replaced'].sum())}")
print(f"Labels changed in the second pass: {int(kept_df['qa_flag'].sum())}")
display(kept_df["label_gold"].value_counts(dropna=False))

# Agreement uses only the original sample rows, which were labelled in both passes.
if SECOND_PASS_COMPLETE and unlabelled_count == 0:
    both_passes = kept_df.index < initial_sample_size
    agreement = 1 - kept_df.loc[both_passes, "qa_flag"].mean()
    kappa = cohen_kappa_score(
        first_pass_labels[kept_df.index[both_passes]],
        kept_df.loc[both_passes, "label_gold"],
    )
    print(f"First/second pass agreement: {agreement:.1%} (Cohen's kappa {kappa:.3f})")

Labelled: 300 / 300
Wrong-company headlines replaced: 0
Labels changed in the second pass: 0


label_gold
positive    152
neutral      83
negative     65
Name: count, dtype: int64

In [22]:
# Save progress using the gold-label schema from the specification.
gold_labels_df = labelling_df.loc[~labelling_df["replaced"], GOLD_COLUMNS].copy()
assert len(gold_labels_df) == initial_sample_size

gold_labels_path.parent.mkdir(parents=True, exist_ok=True)
gold_labels_df.to_csv(gold_labels_path, index=False)

print(f"Saved label progress to {gold_labels_path}")

Saved label progress to C:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks\data\processed\gold_labels.csv


In [23]:
if unlabelled_count > 0:
    print("Labelling is not complete. Finish the blank labels.")
elif not SECOND_PASS_COMPLETE:
    print("First pass complete. Finish the second pass before the final evaluation.")
else:
    print("Labelling complete: gold_labels.csv is ready for model evaluation.")

First pass complete. Finish the second pass before the final evaluation.
